In [1]:

from pathlib import Path
import pandas as pd
import joblib
import json

# Project directory
PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"

# Load exported datasets
authLogsDF = pd.read_csv(DATA_DIR / "authlogs.csv")
coreMLDF = pd.read_csv(DATA_DIR / "core_ml_data.csv")
testMLDF = pd.read_csv(DATA_DIR / "test_ml_data.csv")

# Load trained model
coreModel = joblib.load(
    ARTIFACT_DIR / "isolation_forest.joblib"
)

# Load feature names
with open(
    ARTIFACT_DIR / "feature_columns.json",
    "r",
    encoding="utf-8"
) as f:
    coreFeatureColumns = json.load(f)

print("All files loaded successfully!")
print("Authentication logs:", authLogsDF.shape)
print("Core ML data:", coreMLDF.shape)
print("Test ML data:", testMLDF.shape)
print("Number of model features:", len(coreFeatureColumns))


All files loaded successfully!
Authentication logs: (3000, 40)
Core ML data: (1379, 12)
Test ML data: (500, 27)
Number of model features: 7


In [2]:

def inspect_dataset(df, dataset_name):
    print(f"DATASET: {dataset_name}")
    print("-" * 40)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing cells:", df.isnull().sum().sum())
    print("\nColumn data types:")
    print(df.dtypes)
    print("\nFirst 5 rows:")
    display(df.head())


inspect_dataset(authLogsDF, "PARICHAY Authentication Logs")


DATASET: PARICHAY Authentication Logs
----------------------------------------
Rows: 3000
Columns: 40
Duplicate rows: 0
Missing cells: 4573

Column data types:
AuthType                  object
BrowserID                 object
City                      object
Country                   object
EmailId                   object
EventID                   object
EventName                 object
EventType                 object
IPAddress                 object
Lang                      object
Location                  object
LoginAlert                object
MobileNo                 float64
MultiFactorType           object
OTPSendOn                 object
OTPType                   object
Reason                    object
RememberDevice            object
ServiceId                 object
ServiceName               object
ServiceState              object
SessionID                 object
State                     object
TimeStamp                 object
TokenID                   object
UId            

,AuthType,BrowserID,City,Country,EmailId,EventID,EventName,EventType,IPAddress,Lang,...,departmentName,serviceGroupId,month,user fingerprinting,device fingerprinting,Latitude,Longitude,IsVPN,IsProxy,IPReputationScore
0,Password,BROWSER_007,Kolkata,India,user_197@example.com,EV2801,Login,Failure,10.20.2.10,en,...,IT,1.0,Sep,UF_8EB9FAB8D39B,DEVICE_016,22.549211,88.329549,No,No,82
1,OAuth,BROWSER_029,Mumbai,India,user_33@example.com,EV2190,Login,Success,10.20.0.3,hi,...,Sales,10.0,Oct,UF_D019D721DDE0,DEVICE_006,19.064294,72.875276,Yes,No,65
2,Biometric,BROWSER_006,Delhi,India,user_21@example.com,EV2817,Login,Failure,10.20.1.2,hi,...,HR,7.0,Apr,UF_E769EA64EDB5,DEVICE_031,28.639385,77.219436,No,No,98
3,Password,BROWSER_006,Hyderabad,India,user_86@example.com,EV1251,Password Reset,Success,10.20.3.16,hi,...,Finance,10.0,Jun,UF_7D8CAC73FE4F,DEVICE_012,17.398749,78.497191,No,Yes,41
4,OAuth,BROWSER_010,Pune,India,user_61@example.com,EV1766,Login,Success,10.20.1.25,en,...,Support,NaN,Sep,UF_499,DEVICE_023,18.503030,73.873614,No,No,90


In [4]:
#Missing-value report

def missing_value_report(df):
    report = pd.DataFrame({
        "Column": df.columns,
        "MissingCount": df.isnull().sum().values,
        "MissingPercentage": (
            df.isnull().mean().values * 100
        ).round(2)
    })

    return report.sort_values(
        "MissingPercentage",
        ascending=False
    ).reset_index(drop=True)


quality_report = missing_value_report(authLogsDF)

display(quality_report)


,Column,MissingCount,MissingPercentage
0,MobileNo,951,31.70
1,UserId,951,31.70
2,serviceGroupId,951,31.70
3,OTPType,681,22.70
4,OTPSendOn,546,18.20
5,MultiFactorType,493,16.43
6,AuthType,0,0.00
7,TokenID,0,0.00
8,UId,0,0.00
9,UserAgent,0,0.00


In [5]:

# Investigate missing values by event type

columns_to_check = [
    "MobileNo",
    "UserId",
    "serviceGroupId",
    "OTPType",
    "OTPSendOn",
    "MultiFactorType"
]

missing_by_event = (
    authLogsDF
    .groupby("EventName")[columns_to_check]
    .agg(lambda column: column.isna().sum())
)

print("Missing values by EventName:")
display(missing_by_event)

print("\nTotal records by EventName:")
display(authLogsDF["EventName"].value_counts().to_frame("RecordCount"))


Missing values by EventName:


,MobileNo,UserId,serviceGroupId,OTPType,OTPSendOn,MultiFactorType
EventName,,,,,,
Account Unlock,0,0,0,122,99,85
Login,951,951,951,146,129,91
Logout,0,0,0,147,87,110
OTP Verification,0,0,0,135,127,99
Password Reset,0,0,0,131,104,108



Total records by EventName:


,RecordCount
EventName,
Login,1379
OTP Verification,433
Password Reset,416
Logout,401
Account Unlock,371


In [7]:
#Automated quality report

import pandas as pd

def generate_quality_report(df):
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = int(df.isna().sum().sum())

    summary = {
        "Total_Rows": int(df.shape[0]),
        "Total_Columns": int(df.shape[1]),
        "Missing_Cells": missing_cells,
        "Missing_Percentage": round(
            (missing_cells / total_cells) * 100, 2
        ) if total_cells else 0,
        "Duplicate_Rows": int(df.duplicated().sum()),
        "Duplicate_EventIDs": (
            int(df["EventID"].duplicated().sum())
            if "EventID" in df.columns else None
        ),
        "Invalid_Timestamps": (
            int(pd.to_datetime(
                df["TimeStamp"], errors="coerce", dayfirst=True
            ).isna().sum())
            if "TimeStamp" in df.columns else None
        )
    }

    return summary


quality_summary = generate_quality_report(authLogsDF)

display(pd.DataFrame(
    quality_summary.items(),
    columns=["Metric", "Value"]
))


,Metric,Value
0,Total_Rows,3000.00
1,Total_Columns,40.00
2,Missing_Cells,4573.00
3,Missing_Percentage,3.81
4,Duplicate_Rows,0.00
5,Duplicate_EventIDs,1000.00
6,Invalid_Timestamps,0.00


In [8]:

# Count occurrences of each EventID
event_id_counts = authLogsDF["EventID"].value_counts(
    dropna=False
)

print("Total rows:", len(authLogsDF))
print("Unique EventIDs:", authLogsDF["EventID"].nunique())
print("Repeated EventID values:", (
    authLogsDF["EventID"].value_counts().gt(1).sum()
))

print("\nMost frequently repeated EventIDs:")
display(
    event_id_counts[event_id_counts > 1]
    .head(15)
    .rename("Occurrences")
    .to_frame()
)

print("\nSample rows with repeated EventIDs:")
repeated_ids = event_id_counts[event_id_counts > 1].index

display(
    authLogsDF[
        authLogsDF["EventID"].isin(repeated_ids)
    ][["EventID", "EventName", "EventType", "TimeStamp", "UserName"]]
    .sort_values("EventID")
    .head(20)
)


Total rows: 3000
Unique EventIDs: 2000
Repeated EventID values: 762

Most frequently repeated EventIDs:


,Occurrences
EventID,
EV1000,50
EV2575,5
EV2134,5
EV2017,4
EV2182,4
EV1003,4
EV2015,4
EV2081,4
EV1272,4



Sample rows with repeated EventIDs:


,EventID,EventName,EventType,TimeStamp,UserName
1023,EV1000,Login,Failure,11-07-2026 09:00,user_954
1037,EV1000,OTP Verification,Success,12-07-2026 11:01,user_654
1925,EV1000,Login,Failure,11-07-2026 09:03,user_954
1848,EV1000,Login,Failure,11-07-2026 09:02,user_954
1822,EV1000,OTP Verification,Success,12-07-2026 12:09,user_916
1813,EV1000,OTP Verification,Success,12-07-2026 12:05,user_395
1157,EV1000,Login,Failure,11-07-2026 11:11,user_325
2828,EV1000,OTP Verification,Success,12-07-2026 12:01,user_904
148,EV1000,Login,Failure,11-07-2026 11:14,user_528
1755,EV1000,Login,Success,27-09-2026 23:46,user_858


In [9]:

# Investigate EV1000
ev1000 = authLogsDF[
    authLogsDF["EventID"] == "EV1000"
].copy()

ev1000["ParsedTimestamp"] = pd.to_datetime(
    ev1000["TimeStamp"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

display(
    ev1000[
        [
            "EventID",
            "EventName",
            "EventType",
            "UserName",
            "IPAddress",
            "TimeStamp",
            "ParsedTimestamp"
        ]
    ].sort_values("ParsedTimestamp")
)

print("EV1000 occurrences:", len(ev1000))
print("Distinct users:", ev1000["UserName"].nunique())
print("Distinct IP addresses:", ev1000["IPAddress"].nunique())
print("Distinct timestamps:", ev1000["ParsedTimestamp"].nunique())


,EventID,EventName,EventType,UserName,IPAddress,TimeStamp,ParsedTimestamp
2330,EV1000,Login,Success,user_999,10.20.0.9,10-07-2026 10:00,2026-07-10 10:00:00
84,EV1000,Login,Success,user_999,10.20.2.23,10-07-2026 10:08,2026-07-10 10:08:00
1023,EV1000,Login,Failure,user_954,10.20.0.16,11-07-2026 09:00,2026-07-11 09:00:00
1664,EV1000,Login,Failure,user_954,10.20.0.16,11-07-2026 09:01,2026-07-11 09:01:00
1848,EV1000,Login,Failure,user_954,10.20.0.16,11-07-2026 09:02,2026-07-11 09:02:00
1925,EV1000,Login,Failure,user_954,10.20.0.16,11-07-2026 09:03,2026-07-11 09:03:00
2655,EV1000,Login,Failure,user_954,10.20.0.16,11-07-2026 09:04,2026-07-11 09:04:00
1264,EV1000,Login,Failure,user_954,10.20.3.6,11-07-2026 09:05,2026-07-11 09:05:00
989,EV1000,Login,Failure,user_414,10.20.2.16,11-07-2026 11:10,2026-07-11 11:10:00
1157,EV1000,Login,Failure,user_325,10.20.2.14,11-07-2026 11:11,2026-07-11 11:11:00


EV1000 occurrences: 50
Distinct users: 42
Distinct IP addresses: 35
Distinct timestamps: 47


In [10]:
#save quality report in json

import json
from pathlib import Path

# Create a report containing the measured findings
ai_quality_report = {
    "dataset": "PARICHAY Authentication Logs",
    "total_rows": int(authLogsDF.shape[0]),
    "total_columns": int(authLogsDF.shape[1]),
    "missing_cells": int(authLogsDF.isna().sum().sum()),
    "missing_percentage": round(
        authLogsDF.isna().sum().sum()
        / (authLogsDF.shape[0] * authLogsDF.shape[1]) * 100,
        2
    ),
    "exact_duplicate_rows": int(authLogsDF.duplicated().sum()),
    "unique_event_ids": int(authLogsDF["EventID"].nunique()),
    "repeated_event_id_values": int(
        authLogsDF["EventID"].value_counts().gt(1).sum()
    ),
    "most_repeated_event_id": "EV1000",
    "most_repeated_event_id_count": int(
        (authLogsDF["EventID"] == "EV1000").sum()
    ),
    "invalid_timestamps": int(
        pd.to_datetime(
            authLogsDF["TimeStamp"],
            format="%d-%m-%Y %H:%M",
            errors="coerce"
        ).isna().sum()
    )
}

REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

report_path = REPORT_DIR / "data_quality_report.json"

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(ai_quality_report, file, indent=4)

print("Quality report saved:", report_path.resolve())
print(json.dumps(ai_quality_report, indent=4))


Quality report saved: C:\Users\rishu\Downloads\PARICHAY_AI\reports\data_quality_report.json
{
    "dataset": "PARICHAY Authentication Logs",
    "total_rows": 3000,
    "total_columns": 40,
    "missing_cells": 4573,
    "missing_percentage": 3.81,
    "exact_duplicate_rows": 0,
    "unique_event_ids": 2000,
    "repeated_event_id_values": 762,
    "most_repeated_event_id": "EV1000",
    "most_repeated_event_id_count": 50,
    "invalid_timestamps": 0
}


In [13]:
#loading of json report again

import json

with open(
    PROJECT_DIR / "reports" / "data_quality_report.json",
    "r",
    encoding="utf-8"
) as file:
    quality_data = json.load(file)

print(type(quality_data))
print(quality_data)


print("Total rows:", quality_data["total_rows"])
print("Missing percentage:", quality_data["missing_percentage"])
print("Repeated EventIDs:", quality_data["repeated_event_id_values"])



<class 'dict'>
{'dataset': 'PARICHAY Authentication Logs', 'total_rows': 3000, 'total_columns': 40, 'missing_cells': 4573, 'missing_percentage': 3.81, 'exact_duplicate_rows': 0, 'unique_event_ids': 2000, 'repeated_event_id_values': 762, 'most_repeated_event_id': 'EV1000', 'most_repeated_event_id_count': 50, 'invalid_timestamps': 0}
Total rows: 3000
Missing percentage: 3.81
Repeated EventIDs: 762


In [14]:
def explain_quality_report(report):
    explanations = []

    explanations.append(
        f"The dataset contains {report['total_rows']} rows "
        f"and {report['total_columns']} columns."
    )

    explanations.append(
        f"{report['missing_cells']} cells are missing, "
        f"representing {report['missing_percentage']}% "
        "of all dataset cells."
    )

    explanations.append(
        f"There are {report['exact_duplicate_rows']} "
        "exact duplicate rows."
    )

    explanations.append(
        f"The dataset contains {report['unique_event_ids']} "
        f"unique EventIDs and "
        f"{report['repeated_event_id_values']} EventID values "
        "that occur more than once."
    )

    explanations.append(
        f"{report['most_repeated_event_id']} appears "
        f"{report['most_repeated_event_id_count']} times. "
        "Its reuse should be investigated if EventID is expected "
        "to uniquely identify each event."
    )

    return "\n".join(
        f"{i}. {message}"
        for i, message in enumerate(explanations, start=1)
    )


explanation = explain_quality_report(quality_data)
print(explanation)


1. The dataset contains 3000 rows and 40 columns.
2. 4573 cells are missing, representing 3.81% of all dataset cells.
3. There are 0 exact duplicate rows.
4. The dataset contains 2000 unique EventIDs and 762 EventID values that occur more than once.
5. EV1000 appears 50 times. Its reuse should be investigated if EventID is expected to uniquely identify each event.


In [15]:
#it is public demo api call
import requests

url = "https://jsonplaceholder.typicode.com/todos/1"

response = requests.get(url, timeout=15)

print("Status code:", response.status_code)
print("Request successful:", response.ok)
print("Response data:")
print(response.json())


Status code: 200
Request successful: True
Response data:
{'userId': 1, 'id': 1, 'title': 'delectus aut autem', 'completed': False}


In [16]:
#post request
import requests

url = "https://jsonplaceholder.typicode.com/posts"

payload = {
    "title": "PARICHAY Data Quality Report",
    "body": "Missing values: 3.81%. Repeated EventID values: 762.",
    "userId": 1
}

headers = {
    "Content-Type": "application/json"
}

try:
    response = requests.post(
        url,
        json=payload,
        headers=headers,
        timeout=15
    )

    response.raise_for_status()

    print("Status code:", response.status_code)
    print("Response:", response.json())

except requests.exceptions.Timeout:
    print("Request timed out.")

except requests.exceptions.RequestException as error:
    print("Request failed:", error)


Status code: 201
Response: {'title': 'PARICHAY Data Quality Report', 'body': 'Missing values: 3.81%. Repeated EventID values: 762.', 'userId': 1, 'id': 101}


In [17]:
# Temporary environment variable
import os

os.environ["AI_API_KEY"] = "demo-key-not-real"

print("Demo API key configured:", "AI_API_KEY" in os.environ)


Demo API key configured: True


In [18]:
#read variable

import os

api_key = os.getenv("AI_API_KEY")

print("API key available:", api_key is not None)


API key available: True


In [1]:
#verifying a dummy variable
import os

api_key = os.getenv("OPENAI_API_KEY")

print("Environment variable loaded:", api_key is not None)
print("Key is configured:", bool(api_key))


Environment variable loaded: True
Key is configured: True


In [1]:
import os

api_key = os.getenv("OPENAI_API_KEY")
print("Real key configured:", bool(api_key and api_key != "demo-key-not-real"))

Real key configured: True


In [2]:
%pip install -U openai

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 783.3 kB/s eta 0:00:03
   --------- ------------------------------ 0.5/2.2 MB 783.3 kB/s eta 0:00:03
   -------------- ------------------------- 0.8/2.2 MB 839.0 kB/s eta 0:00:02
   ------------------ --------------------- 1.0/2.2 MB 796.1 kB/s eta 0:00:02
   ------------------ --------------------- 1.0/2.2 MB 796.1 kB/s eta 0:00:02
   ----------------------- ---------------- 1.3/2.2 MB 833.1 kB/s eta 0:00:02
   ---------------------------- ----------- 1.6/2.2 MB 884.7 kB/s eta 0:00:01
   -------------------------------- ------- 1.8/2.2 MB 844.0 kB/s eta 0:00:01
   -------------------------------- ------- 1.8/2.2 MB 844.0 kB/s eta 0:00:01
   ------------------------------------- -- 2.1/2.2 MB 807.9 kB/s eta 0:00:01
   ----------

In [4]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\rishu\Downloads\PARICHAY_AI")

print("Project directory exists:", PROJECT_DIR.exists())
print("Report exists:", (PROJECT_DIR / "reports" / "data_quality_report.json").exists())

Project directory exists: True
Report exists: True


In [ ]:
import os
import json
from openai import OpenAI

api_key = os.getenv("OPENAI_API_KEY")

if not api_key or api_key == "demo-key-not-real":
    raise ValueError("A valid OPENAI_API_KEY is required.")

client = OpenAI(api_key=api_key)

with open(
    PROJECT_DIR / "reports" / "data_quality_report.json",
    "r",
    encoding="utf-8"
) as file:
    quality_data = json.load(file)

prompt = f"""
You are a data quality analyst.

Analyze this PARICHAY authentication log quality report:
{json.dumps(quality_data, indent=2)}

Explain:
1. The key findings.
2. Which issues need investigation.
3. Recommended next checks.

Use simple language. Do not claim that repeated EventIDs prove
a security attack. Do not invent facts beyond the supplied metrics.
"""

response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

print(response.output_text)
